# Double Machine Learning (simple version)

**Project: EconoCausal. Question: does sending an email change customer behaviour?**

**The idea in one example:** suppose customers like Asha buy 8% of the time with no email and 11% of the time with an email. Her *uplift* is 11% - 8% = **3 percentage points**. We never see Asha both ways, so the model estimates the uplift for *similar profiles* of customers (this is called the CATE, or conditional average treatment effect).

**What we use**
- `X` = what we knew about the customer **before** the campaign (from notebook 02).
- `T` = what they got: `0` = no email, `1` = men's email, `2` = women's email. These are categories, not discount sizes.
- `Y` = what happened: `visit` (main outcome), `conversion` (bought), `spend` (dollars).

**What this notebook does**
1. Load the data prepared by notebook 02 (70% train, 30% held out).
2. Train one causal model per outcome on the training rows only.
3. Predict the uplift of each email for the held-out customers.
4. Check: does ranking customers by predicted uplift beat random targeting? (uplift / Qini curve)
5. Save scores, charts and models.

**Before you run:** run `02_preprocessing_feature_engineering.ipynb` first (Run All). Training takes a few minutes. Needs: numpy, pandas, scikit-learn, econml, matplotlib, joblib.

**Limits:** these are *email* effects (no discount data exists), and estimates are for profiles, not certain facts about one person.

## Step 1: Imports and folders

In [ ]:
from pathlib import Path
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from econml.dml import CausalForestDML

SEED = 42
OUTCOMES = ["visit", "conversion", "spend"]
EMAILS = {1: "mens", 2: "womens"}          # 0 = no email is the comparison group

# Find the project folder (the one that contains data/raw/hillstrom.csv)
folders = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in folders if (p / "data/raw/hillstrom.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open the project folder that contains data/raw/hillstrom.csv")

OUTPUT = ROOT / "results/double_ml"       # tables and charts
MODEL_DIR = ROOT / "models/double_ml"     # saved models
OUTPUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## Step 2: Load the data from notebook 02

We use the **original randomized** training and test files (not the matched rows from notebook 04). The 30% test set is only used at the end to check the model. It is never used for training.

In [ ]:
needed = ["data/processed/causal_train.csv", "data/processed/causal_test.csv",
          "data/processed/causal_clean.csv", "models/causal_preprocessor.joblib",
          "results/causal_preprocessing_summary.json"]
missing = [p for p in needed if not (ROOT / p).exists()]
if missing:
    raise FileNotFoundError("Run notebook 02 first. Missing: " + ", ".join(missing))

train = pd.read_csv(ROOT / needed[0])
test = pd.read_csv(ROOT / needed[1]).sort_values("customer_id").reset_index(drop=True)
clean = pd.read_csv(ROOT / needed[2])
encoder = joblib.load(ROOT / needed[3])      # only load files you created yourself
prep = json.loads((ROOT / needed[4]).read_text())

features = prep["feature_columns_after_encoding"]       # the X columns
raw_features = prep["feature_columns_before_encoding"]

# Safety checks: X must not contain the answer, and train/test must not overlap
leaks = {"customer_id", "segment", "treatment", "treatment_name", *OUTCOMES}
assert not leaks.intersection(features), "X contains treatment or outcome columns"
assert set(train.customer_id).isdisjoint(test.customer_id), "train and test overlap"

X_train = train[features].to_numpy(dtype=float)
X_test = test[features].to_numpy(dtype=float)
T_train = train["treatment"].to_numpy()

print(f"Training rows: {len(train):,} | held-out rows: {len(test):,} | features: {len(features)}")

## Step 3: How Double ML works (plain English)

Think of two helpers and one judge.

1. **Outcome helper** (Random Forest): learns the *typical* outcome for each customer profile, ignoring the email.
2. **Email helper** (Random Forest): learns the *chance* of getting each email for each profile. In a randomized test this is about one third each.
3. **Causal forest (the judge):** looks at what is left over after the helpers have done their part, and learns how the email changes the outcome for different profiles.

**Cross-fitting:** the training data is split into 3 folds. Each helper predicts a fold using models trained on the *other* folds, so no helper predicts a row it has already seen. This avoids overfitting.

**Honest forest:** the forest uses different rows to decide how to split and to estimate effects. Large leaves (`min_samples_leaf`) keep estimates stable, because purchases are rare.

DML does **not** remove hidden confounders by itself. Our results rely on the campaign being randomized.

In [ ]:
def make_model():
    outcome_helper = RandomForestRegressor(n_estimators=60, max_depth=10, min_samples_leaf=40,
                                           random_state=SEED, n_jobs=2)
    email_helper = RandomForestClassifier(n_estimators=60, max_depth=8, min_samples_leaf=80,
                                          random_state=SEED, n_jobs=2)
    return CausalForestDML(
        model_y=outcome_helper,            # predicts the outcome
        model_t=email_helper,              # predicts which email was sent
        discrete_outcome=False,            # also fine for 0/1 outcomes (predicts a probability)
        discrete_treatment=True,           # the 3 actions are categories
        categories=[0, 1, 2],
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),   # cross-fitting
        n_estimators=120, subforest_size=4, min_samples_leaf=120, max_depth=10,
        honest=True, inference=True,       # inference=True gives confidence intervals
        random_state=SEED, n_jobs=2)

models = {}                                                        # one model per outcome
test_scores = test[["customer_id", "treatment", *OUTCOMES]].copy() # held-out predictions go here

for outcome in OUTCOMES:
    start = time.time()
    print(f"Training {outcome}...", flush=True)
    model = make_model()
    model.fit(train[outcome].to_numpy(), T_train, X=X_train)       # training rows only
    models[outcome] = model

    for code_number, name in EMAILS.items():
        # estimated uplift of this email vs no email, for each held-out customer
        effect = model.effect(X_test, T0=0, T1=code_number).ravel()
        low, high = model.effect_interval(X_test, T0=0, T1=code_number, alpha=0.05)
        test_scores[f"{outcome}_{name}"] = effect
        test_scores[f"{outcome}_{name}_lower95"] = low.ravel()
        test_scores[f"{outcome}_{name}_upper95"] = high.ravel()

    joblib.dump(model, MODEL_DIR / f"{outcome}_forest.joblib", compress=3)
    print(f"  done in {time.time() - start:.0f} seconds")

## Step 4: What the scores mean

- `visit_mens = 0.04` means: for that profile, the men's email is estimated to raise the visit chance by **4 percentage points** versus no email.
- A negative score means the email is estimated to *lower* the outcome. Keep it, it is not an error.
- `conversion` is also a probability. `spend` is extra **dollars per customer** (not profit).
- Each score has a 95% interval (`_lower95`, `_upper95`). Wide intervals mean we are unsure.

**Sanity check:** because the test set is randomized, a simple difference in group averages is a fair estimate of the *average* effect. The average of our predicted scores should be close to it.

In [ ]:
rows = []
for outcome in OUTCOMES:
    for code_number, name in EMAILS.items():
        emailed = test.loc[test.treatment == code_number, outcome]
        control = test.loc[test.treatment == 0, outcome]
        diff = emailed.mean() - control.mean()
        se = np.sqrt(emailed.var(ddof=1) / len(emailed) + control.var(ddof=1) / len(control))
        rows.append({"outcome": outcome, "email": name,
                     "simple_difference": diff,
                     "diff_lower95": diff - 1.96 * se,
                     "diff_upper95": diff + 1.96 * se,
                     "mean_predicted_uplift": test_scores[f"{outcome}_{name}"].mean()})
averages = pd.DataFrame(rows)
averages.round(5)

## Step 5: Does ranking by predicted uplift beat random?

A good model should find the customers who benefit *most* from the email.

**How the check works**
1. Sort held-out customers from highest predicted uplift to lowest.
2. Walk down the list. At each point, count the extra outcomes of emailed customers compared with not-emailed customers (each row counts `+2 x outcome` if emailed and `-2 x outcome` if not; the 2 is because each group is about half of the pair).
3. Plot the running total (the **gain curve**) and compare it with the **random line** (what you would get by picking customers in no special order).
4. The **Qini area** is the area between the two lines. **Bigger and positive is better.**

**Uncertainty:** we resample the held-out rows 1,000 times (bootstrap) to get an approximate 95% interval. If the interval is entirely above 0, there is some evidence the ranking helps. If it includes 0, we cannot tell.

Tied scores are handled fairly (equal scores are cut together, so row order cannot create a fake advantage).

In [ ]:
def gain_curve(y, got_email, score):
    """Running extra outcome per eligible customer, when contacting the top-ranked first."""
    y, got_email, score = map(np.asarray, (y, got_email, score))
    order = np.argsort(-score, kind="stable")              # highest predicted uplift first
    score_sorted = score[order]
    contribution = 2 * y[order] * (2 * got_email[order] - 1)
    ends = np.r_[np.flatnonzero(np.diff(score_sorted) != 0), len(y) - 1]   # end of each tie group
    fraction = np.r_[0.0, (ends + 1) / len(y)]
    gain = np.r_[0.0, np.cumsum(contribution)[ends] / len(y)]
    random_gain = fraction * gain[-1]                       # straight line to the same endpoint
    excess = gain - random_gain
    area = float(np.sum(np.diff(fraction) * (excess[1:] + excess[:-1]) / 2))
    return fraction, gain, random_gain, area

# Tiny sanity checks: equal scores give no advantage, a perfect ranking gives a positive area
assert abs(gain_curve([1, 0, 1, 0], [1, 0, 1, 0], np.ones(4))[3]) < 1e-12
assert gain_curve([1, 0, 0, 0], [1, 0, 1, 0], [4, 3, 2, 1])[3] > 0

BOOTSTRAPS = 1000
curves = {}
ranking_rows = []

for outcome in OUTCOMES:
    curves[outcome] = {}
    for code_number, name in EMAILS.items():
        # keep only this email group and the no-email group
        keep = test.treatment.isin([0, code_number]).to_numpy()
        y = test.loc[keep, outcome].to_numpy()
        got_email = (test.loc[keep, "treatment"] == code_number).astype(int).to_numpy()
        score = test_scores.loc[keep, f"{outcome}_{name}"].to_numpy()

        fraction, gain, random_gain, area = gain_curve(y, got_email, score)

        rng = np.random.default_rng(SEED)
        areas = np.empty(BOOTSTRAPS)
        for i in range(BOOTSTRAPS):
            pick = rng.integers(0, len(y), len(y))
            areas[i] = gain_curve(y[pick], got_email[pick], score[pick])[3]
        low, high = np.quantile(areas, [0.025, 0.975])

        if low > 0:
            verdict = "Evidence the ranking helps (exploratory)"
        elif high < 0:
            verdict = "Evidence the ranking hurts (exploratory)"
        else:
            verdict = "Cannot tell"
        ranking_rows.append({"outcome": outcome, "email": name, "rows": len(y),
                             "qini_area": area, "lower95": low, "upper95": high, "verdict": verdict})

        grid = np.linspace(0, 1, 101)
        curves[outcome][name] = {"fraction": grid.tolist(),
                                 "gain": np.interp(grid, fraction, gain).tolist(),
                                 "random": np.interp(grid, fraction, random_gain).tolist()}

rankings = pd.DataFrame(ranking_rows)
rankings.round(6)

## Step 6: Charts

- Horizontal axis: fraction of customers contacted (0 to 100%), best predicted first.
- **Left:** solid line = our ranking, dashed line = random. Higher is better.
- **Right:** the gap between the two (`gain - random`). Above 0 means the ranking helped.

A high curve for *visits* does not prove more purchases or revenue. Check each outcome separately. Do not pick the "best fraction" from this test set and then claim it as a proven policy; that needs separate validation data.

In [ ]:
for outcome in OUTCOMES:
    fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
    for name in EMAILS.values():
        c = curves[outcome][name]
        x, gain, base = np.array(c["fraction"]), np.array(c["gain"]), np.array(c["random"])
        left.plot(x, gain, label=f"{name}: our ranking")
        left.plot(x, base, linestyle="--", label=f"{name}: random")
        right.plot(x, gain - base, label=name)
    left.set(title=f"{outcome}: cumulative gain (held-out)", xlabel="Fraction of customers contacted",
             ylabel="Extra outcome per eligible customer")
    right.set(title=f"{outcome}: gain minus random", xlabel="Fraction of customers contacted",
              ylabel="Gain minus random")
    right.axhline(0, color="grey", linestyle="--")
    left.legend(fontsize=8)
    right.legend(fontsize=8)
    fig.savefig(OUTPUT / f"{outcome}_uplift.png", dpi=140)
    plt.show()

## Step 7: Save everything

- `test_scores.csv`: predictions for the **held-out** customers. **Use this one for honest evaluation.**
- `customer_scores.csv`: predictions for all customers. Rows with `split == "train"` are *in-sample* (the model saw them), so never present them as independent validation.
- `uplift_curves.json`: chart numbers for the dashboard.
- Models are saved under `models/double_ml/`. Only load joblib files you created yourself.

In [ ]:
# Predictions for ALL customers (train rows are in-sample)
X_all = encoder.transform(clean[raw_features])
all_scores = clean[["customer_id", "treatment", *OUTCOMES]].copy()
all_scores["split"] = np.where(clean.customer_id.isin(train.customer_id), "train", "test")
for outcome in OUTCOMES:
    for code_number, name in EMAILS.items():
        all_scores[f"{outcome}_{name}"] = models[outcome].effect(X_all, T0=0, T1=code_number).ravel()

all_scores.to_csv(OUTPUT / "customer_scores.csv", index=False)
test_scores.to_csv(OUTPUT / "test_scores.csv", index=False)
averages.to_csv(OUTPUT / "average_effects.csv", index=False)
rankings.to_csv(OUTPUT / "ranking_evaluation.csv", index=False)
(OUTPUT / "uplift_curves.json").write_text(json.dumps(curves, indent=2))

joblib.dump({"encoder": encoder, "models": models, "raw_features": raw_features,
             "encoded_features": features, "treatments": {0: "No E-Mail", 1: "Mens E-Mail", 2: "Womens E-Mail"}},
            MODEL_DIR / "double_ml_bundle.joblib", compress=3)

for r in ranking_rows:
    print(f"{r['outcome']} / {r['email']}: {r['verdict']}  "
          f"(Qini area {r['qini_area']:.5f}, 95% interval {r['lower95']:.5f} to {r['upper95']:.5f})")
print("Saved to", OUTPUT)

## What to tell your manager

"We used customer information from before the campaign. One model learned typical outcomes and another learned who got which email. A causal forest then estimated how much each email changes visits, purchases and spending for similar customers. We kept 30% of customers out of training and checked whether ranking them by predicted benefit beats random targeting. The tables and charts show where the evidence is clear and where it is not. These are *email* effects, not discount effects, and not certain outcomes for any one person."

## How to read the result honestly

- **Average effect vs. ranking:** an email can help on average (Step 4) even if the model cannot reliably tell *who* benefits most (Step 5). These are two different questions.
- Six outcome/email checks are exploratory and are **not** adjusted for multiple testing. A lower bound barely above zero is weak evidence.
- The intervals ignore retraining variation, and the earlier EDA looked at the full dataset. Stronger claims need a fresh validation sample or a new experiment.
- No profit or discount conclusion follows from this notebook.

## Next steps

DoWhy refutation / sensitivity checks (mid-project notebook), then budget optimization (Week 3). The dashboard can load `test_scores.csv` and `uplift_curves.json`.